In [1]:
import itertools
import math
from sympy import *
import time

In this notebook we search for common factors of two 'collision polynomials' $p(r,s)=r^a+s^tr^b-s^ur^c-s^vr^d$ and $q(r,s)=r^e+s^xr^f-s^yr^g-s^zr^h$.

The following function generates a list of 'ignorable' GCDs, which when set equal to $0$ force one or both of $r,s$ to be irrational, negative, or equal to $1$, or force $s$ to be a rational power of $r$. These were encountered in initial exploratory analysis, and are now preemptively enumerated and factored out in order to focus on genuinely problematic GCDs.

In [2]:
def build_divisors(l):
    r, s = symbols('r s')
    divisors = [ r**6 + r**5 + r**4 + r**3 + r**2 + r + 1, r**4 + r**3 + r**2 + r + 1,
                r**4 - r**2 + 1, r**4 - r**3 + r**2 - r + 1, r**4 + 1, r**2 - r + 1, r**2+r+1,
                 r**2*s**2 + r*s - s - 1, r**2+1, s**2-r, r*s**2-1, s+r+1, (r+1)*s+1, r*s+r+s,
                  (r**2+r)*s+1, r**2+r*s+s, r-1, r+1, s**2+1, s**2+r, r*s**2+1, r**2+r+s,
                 r*s+r+1, r**2*s+r+1, r**2 + r*s + 1, r**2*s + r + s  ]
    for i in range(l):
        divisors.append(s-r**i)
        divisors.append(s+r**i)
    for i in range(1,l):
        divisors.append(s*r**i+1)
        divisors.append(s*r**i-1)
    return divisors

The following function factors out the ignorable GCDs enumerated in the previous function.

In [3]:
def factor_out(divisors,a,b,c,d,e,f,g,h,t,u,v,x,y,z,l):
    r, s = symbols('r s')
    p = r**(a)+s**t*r**(b)-s**u*r**(c)-s**v*r**(d)
    q = r**(e)+s**x*r**(f)-s**y*r**(g)-s**z*r**(h)

    G = gcd(p,q)

    for div in divisors:
        quot, rem = reduced(G, [div], [r, s])
        if rem == 0:
            p = (p/div).cancel()
            q = (q/div).cancel()

    return p,q

Below is the good_octuple function mentioned in the paper, which prevents redundancy and degeneracy in solutions to $p(r,s)=q(r,s)=0$.

In [4]:
def good_octuple(a,b,c,d,e,f,g,h,t,u,v,x,y,z):
    good = True
    if not (min(a,b,c,d)==min(e,f,g,h)==0):
        good = False
    if (u==0 and a==c) or (v==0 and a==d) or (t==u and b==c) or (t==v and b==d):
        good = False
    if (y==0 and e==g) or (z==0 and e==h) or (x==y and f==g) or (x==z and f==h):
        good = False
    if (t==0 and a<b) or (u==v and c<d) or (x==0 and e<f) or (y==z and g<h) or (u==0 and t==v and c>=a) or (y==0 and x==z and e<=g):
        good = False

    if (t==x==0 and {a,b}=={e,f}) or (u==v==y==z and {c,d}=={g,h}):
        good = False
    if ((0,t) == (u,v) == (0,x) == (y,z)) and (((a,b) == (e,f) and (c,d) == (g, h)) or ((a,b) == (g,h) and (c,d) == (e,f))):
        good = False

    if ((t,u,v)==(x,y,z) and a-e==b-f==c-g==d-h):
        good = False

    return good

Below is the set $T_2$ from the paper holding the possible values of $(t,u,v)$ and $(x,y,z)$. The first four elements form $T_1$.

In [5]:
ExponentTuples= [(0,1,1),(0,0,1),(1,1,1),(1,0,1),(0,2,2),(0,0,2),(2,2,2),(2,0,2),(1,1,2),(2,1,1),(1,2,2),(2,1,2),(0,1,2),(1,0,2)]

We create dictionaries to store problematic GCDs for two and three-row collisions, respectively.

In [6]:
two_row_GCD_dict = {}
for i in range(4):
        for j in range(i,4):
            (t,u,v) = ExponentTuples[i]
            (x,y,z) = ExponentTuples[j]
            two_row_GCD_dict[(t,u,v,x,y,z)] = []

In [7]:
three_row_GCD_dict = {}
for i in range(14):
        for j in range(i,14):
            (t,u,v) = ExponentTuples[i]
            (x,y,z) = ExponentTuples[j]
            three_row_GCD_dict[(t,u,v,x,y,z)] = []

The following is our implementation of the PossibleGCDs algorithm from the paper, with the additional step of preemptively dividing out known ignorable GCDs.

In [7]:
def GCDs(rows, l, sixtuples):
    start = time.time()
    done = 0
    L = range(l)
    divisors = build_divisors(l)
    r, s = symbols('r s')
    for (t,u,v,x,y,z) in sixtuples:
        remaining = []
        for (a,b,c,d,e,f,g,h) in itertools.product(L,L,L,L,L,L,L,L):
            if good_octuple(a,b,c,d,e,f,g,h,t,u,v,x,y,z):
                p,q = factor_out(divisors,a,b,c,d,e,f,g,h,t,u,v,x,y,z,l)
                D = gcd(p,q)
                if not D.is_constant():
                    remaining.append((a,b,c,d,e,f,g,h,D))
        if rows == 2:
          two_row_GCD_dict[(t,u,v,x,y,z)] = remaining
        if rows == 3:
          three_row_GCD_dict[(t,u,v,x,y,z)] = remaining
        done += 1
        print(done)
    end = time.time()
    print('Total time elapsed (in sec.):', round(end-start,2))

Executing the function separately for two-row and three-row collisions below, we find that the only non-ignorable GCDs are those corresponding to the double collisions discussed in the paper, all of which arise from some transformation of the system $1+r^2=s+rs$, $1+r^2=r+s$, satisfied when $s=r^2-r+1$.

In [9]:
GCDs(3,4,three_row_GCD_dict.keys())

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
Total time elapsed (in sec.): 16460.69


In [10]:
three_row_GCD_dict

{(0, 1, 1, 0, 1, 1): [],
 (0, 1, 1, 0, 0, 1): [(3, 0, 1, 0, 2, 0, 1, 0, r**2 - r - s + 1),
  (3, 0, 2, 1, 2, 0, 1, 1, r**2 - r*s - r + 1),
  (3, 0, 3, 2, 2, 0, 1, 2, r**2*s - r**2 + r - 1)],
 (0, 1, 1, 1, 1, 1): [(1, 0, 3, 0, 0, 1, 2, 0, r**2*s - r*s + s - 1),
  (2, 1, 3, 0, 1, 1, 2, 0, r**2*s - r*s - r + s),
  (3, 2, 3, 0, 2, 1, 2, 0, r**2*s - r**2 - r*s + s)],
 (0, 1, 1, 1, 0, 1): [],
 (0, 1, 1, 0, 2, 2): [],
 (0, 1, 1, 0, 0, 2): [],
 (0, 1, 1, 2, 2, 2): [],
 (0, 1, 1, 2, 0, 2): [],
 (0, 1, 1, 1, 1, 2): [],
 (0, 1, 1, 2, 1, 1): [],
 (0, 1, 1, 1, 2, 2): [],
 (0, 1, 1, 2, 1, 2): [],
 (0, 1, 1, 0, 1, 2): [],
 (0, 1, 1, 1, 0, 2): [],
 (0, 0, 1, 0, 0, 1): [],
 (0, 0, 1, 1, 1, 1): [],
 (0, 0, 1, 1, 0, 1): [],
 (0, 0, 1, 0, 2, 2): [],
 (0, 0, 1, 0, 0, 2): [],
 (0, 0, 1, 2, 2, 2): [],
 (0, 0, 1, 2, 0, 2): [],
 (0, 0, 1, 1, 1, 2): [],
 (0, 0, 1, 2, 1, 1): [],
 (0, 0, 1, 1, 2, 2): [],
 (0, 0, 1, 2, 1, 2): [],
 (0, 0, 1, 0, 1, 2): [],
 (0, 0, 1, 1, 0, 2): [],
 (1, 1, 1, 1, 1, 1): [],
 (1, 1, 1,

In [8]:
GCDs(2,8,two_row_GCD_dict.keys())

1
2
3
4
5
6
7
8
9
10
Total time elapsed (in sec.): 53995.95


In [9]:
two_row_GCD_dict

{(0, 1, 1, 0, 1, 1): [],
 (0, 1, 1, 0, 0, 1): [(3, 0, 1, 0, 2, 0, 1, 0, r**2 - r - s + 1),
  (3, 0, 2, 1, 2, 0, 1, 1, r**2 - r*s - r + 1),
  (3, 0, 3, 2, 2, 0, 1, 2, r**2*s - r**2 + r - 1),
  (3, 0, 4, 3, 2, 0, 1, 3, r**3*s - r**2 + r - 1),
  (3, 0, 5, 4, 2, 0, 1, 4, r**4*s - r**2 + r - 1),
  (3, 0, 6, 5, 2, 0, 1, 5, r**5*s - r**2 + r - 1),
  (3, 0, 7, 6, 2, 0, 1, 6, r**6*s - r**2 + r - 1),
  (4, 1, 1, 0, 3, 1, 2, 0, r**3 - r**2 + r - s),
  (5, 2, 1, 0, 4, 2, 3, 0, r**4 - r**3 + r**2 - s),
  (6, 0, 2, 0, 4, 0, 2, 0, r**4 - r**2 - s + 1),
  (6, 0, 3, 1, 4, 0, 2, 1, r**4 - r**2 - r*s + 1),
  (6, 0, 4, 2, 4, 0, 2, 2, r**4 - r**2*s - r**2 + 1),
  (6, 0, 5, 3, 4, 0, 2, 3, r**4 - r**3*s - r**2 + 1),
  (6, 0, 6, 4, 4, 0, 2, 4, r**4*s - r**4 + r**2 - 1),
  (6, 0, 7, 5, 4, 0, 2, 5, r**5*s - r**4 + r**2 - 1),
  (6, 3, 1, 0, 5, 3, 4, 0, r**5 - r**4 + r**3 - s),
  (7, 1, 2, 0, 5, 1, 3, 0, r**5 - r**3 + r - s),
  (7, 4, 1, 0, 6, 4, 5, 0, r**6 - r**5 + r**4 - s)],
 (0, 1, 1, 1, 1, 1): [(1, 0, 3, 0, 

Three-row runtime: 16460 seconds

Two-row runtime 54000 seconds

In [15]:
r, s = symbols('r s')

Three_row_out = {(0, 1, 1, 0, 1, 1): [],
 (0, 1, 1, 0, 0, 1): [(3, 0, 1, 0, 2, 0, 1, 0, r**2 - r - s + 1),
  (3, 0, 2, 1, 2, 0, 1, 1, r**2 - r*s - r + 1),
  (3, 0, 3, 2, 2, 0, 1, 2, r**2*s - r**2 + r - 1)],
 (0, 1, 1, 1, 1, 1): [(1, 0, 3, 0, 0, 1, 2, 0, r**2*s - r*s + s - 1),
  (2, 1, 3, 0, 1, 1, 2, 0, r**2*s - r*s - r + s),
  (3, 2, 3, 0, 2, 1, 2, 0, r**2*s - r**2 - r*s + s)],
 (0, 1, 1, 1, 0, 1): [],
 (0, 1, 1, 0, 2, 2): [],
 (0, 1, 1, 0, 0, 2): [],
 (0, 1, 1, 2, 2, 2): [],
 (0, 1, 1, 2, 0, 2): [],
 (0, 1, 1, 1, 1, 2): [],
 (0, 1, 1, 2, 1, 1): [],
 (0, 1, 1, 1, 2, 2): [],
 (0, 1, 1, 2, 1, 2): [],
 (0, 1, 1, 0, 1, 2): [],
 (0, 1, 1, 1, 0, 2): [],
 (0, 0, 1, 0, 0, 1): [],
 (0, 0, 1, 1, 1, 1): [],
 (0, 0, 1, 1, 0, 1): [],
 (0, 0, 1, 0, 2, 2): [],
 (0, 0, 1, 0, 0, 2): [],
 (0, 0, 1, 2, 2, 2): [],
 (0, 0, 1, 2, 0, 2): [],
 (0, 0, 1, 1, 1, 2): [],
 (0, 0, 1, 2, 1, 1): [],
 (0, 0, 1, 1, 2, 2): [],
 (0, 0, 1, 2, 1, 2): [],
 (0, 0, 1, 0, 1, 2): [],
 (0, 0, 1, 1, 0, 2): [],
 (1, 1, 1, 1, 1, 1): [],
 (1, 1, 1, 1, 0, 1): [],
 (1, 1, 1, 0, 2, 2): [],
 (1, 1, 1, 0, 0, 2): [],
 (1, 1, 1, 2, 2, 2): [],
 (1, 1, 1, 2, 0, 2): [],
 (1, 1, 1, 1, 1, 2): [],
 (1, 1, 1, 2, 1, 1): [],
 (1, 1, 1, 1, 2, 2): [],
 (1, 1, 1, 2, 1, 2): [],
 (1, 1, 1, 0, 1, 2): [],
 (1, 1, 1, 1, 0, 2): [],
 (1, 0, 1, 1, 0, 1): [],
 (1, 0, 1, 0, 2, 2): [],
 (1, 0, 1, 0, 0, 2): [],
 (1, 0, 1, 2, 2, 2): [],
 (1, 0, 1, 2, 0, 2): [],
 (1, 0, 1, 1, 1, 2): [],
 (1, 0, 1, 2, 1, 1): [],
 (1, 0, 1, 1, 2, 2): [],
 (1, 0, 1, 2, 1, 2): [],
 (1, 0, 1, 0, 1, 2): [],
 (1, 0, 1, 1, 0, 2): [],
 (0, 2, 2, 0, 2, 2): [],
 (0, 2, 2, 0, 0, 2): [(3, 0, 1, 0, 2, 0, 1, 0, r**2 - r - s**2 + 1),
  (3, 0, 2, 1, 2, 0, 1, 1, r**2 - r*s**2 - r + 1),
  (3, 0, 3, 2, 2, 0, 1, 2, r**2*s**2 - r**2 + r - 1)],
 (0, 2, 2, 2, 2, 2): [(1, 0, 3, 0, 0, 1, 2, 0, r**2*s**2 - r*s**2 + s**2 - 1),
  (2, 1, 3, 0, 1, 1, 2, 0, r**2*s**2 - r*s**2 - r + s**2),
  (3, 2, 3, 0, 2, 1, 2, 0, r**2*s**2 - r**2 - r*s**2 + s**2)],
 (0, 2, 2, 2, 0, 2): [],
 (0, 2, 2, 1, 1, 2): [],
 (0, 2, 2, 2, 1, 1): [],
 (0, 2, 2, 1, 2, 2): [],
 (0, 2, 2, 2, 1, 2): [],
 (0, 2, 2, 0, 1, 2): [],
 (0, 2, 2, 1, 0, 2): [],
 (0, 0, 2, 0, 0, 2): [],
 (0, 0, 2, 2, 2, 2): [],
 (0, 0, 2, 2, 0, 2): [],
 (0, 0, 2, 1, 1, 2): [],
 (0, 0, 2, 2, 1, 1): [],
 (0, 0, 2, 1, 2, 2): [],
 (0, 0, 2, 2, 1, 2): [],
 (0, 0, 2, 0, 1, 2): [],
 (0, 0, 2, 1, 0, 2): [],
 (2, 2, 2, 2, 2, 2): [],
 (2, 2, 2, 2, 0, 2): [],
 (2, 2, 2, 1, 1, 2): [],
 (2, 2, 2, 2, 1, 1): [],
 (2, 2, 2, 1, 2, 2): [],
 (2, 2, 2, 2, 1, 2): [],
 (2, 2, 2, 0, 1, 2): [],
 (2, 2, 2, 1, 0, 2): [],
 (2, 0, 2, 2, 0, 2): [],
 (2, 0, 2, 1, 1, 2): [],
 (2, 0, 2, 2, 1, 1): [],
 (2, 0, 2, 1, 2, 2): [],
 (2, 0, 2, 2, 1, 2): [],
 (2, 0, 2, 0, 1, 2): [],
 (2, 0, 2, 1, 0, 2): [],
 (1, 1, 2, 1, 1, 2): [],
 (1, 1, 2, 2, 1, 1): [],
 (1, 1, 2, 1, 2, 2): [],
 (1, 1, 2, 2, 1, 2): [],
 (1, 1, 2, 0, 1, 2): [],
 (1, 1, 2, 1, 0, 2): [],
 (2, 1, 1, 2, 1, 1): [],
 (2, 1, 1, 1, 2, 2): [],
 (2, 1, 1, 2, 1, 2): [],
 (2, 1, 1, 0, 1, 2): [],
 (2, 1, 1, 1, 0, 2): [],
 (1, 2, 2, 1, 2, 2): [],
 (1, 2, 2, 2, 1, 2): [],
 (1, 2, 2, 0, 1, 2): [],
 (1, 2, 2, 1, 0, 2): [],
 (2, 1, 2, 2, 1, 2): [],
 (2, 1, 2, 0, 1, 2): [],
 (2, 1, 2, 1, 0, 2): [],
 (0, 1, 2, 0, 1, 2): [],
 (0, 1, 2, 1, 0, 2): [],
 (1, 0, 2, 1, 0, 2): []}


In [10]:
r, s = symbols('r s')

Two_row_out = {(0, 1, 1, 0, 1, 1): [],
 (0, 1, 1, 0, 0, 1): [(3, 0, 1, 0, 2, 0, 1, 0, r**2 - r - s + 1),
  (3, 0, 2, 1, 2, 0, 1, 1, r**2 - r*s - r + 1),
  (3, 0, 3, 2, 2, 0, 1, 2, r**2*s - r**2 + r - 1),
  (3, 0, 4, 3, 2, 0, 1, 3, r**3*s - r**2 + r - 1),
  (3, 0, 5, 4, 2, 0, 1, 4, r**4*s - r**2 + r - 1),
  (3, 0, 6, 5, 2, 0, 1, 5, r**5*s - r**2 + r - 1),
  (3, 0, 7, 6, 2, 0, 1, 6, r**6*s - r**2 + r - 1),
  (4, 1, 1, 0, 3, 1, 2, 0, r**3 - r**2 + r - s),
  (5, 2, 1, 0, 4, 2, 3, 0, r**4 - r**3 + r**2 - s),
  (6, 0, 2, 0, 4, 0, 2, 0, r**4 - r**2 - s + 1),
  (6, 0, 3, 1, 4, 0, 2, 1, r**4 - r**2 - r*s + 1),
  (6, 0, 4, 2, 4, 0, 2, 2, r**4 - r**2*s - r**2 + 1),
  (6, 0, 5, 3, 4, 0, 2, 3, r**4 - r**3*s - r**2 + 1),
  (6, 0, 6, 4, 4, 0, 2, 4, r**4*s - r**4 + r**2 - 1),
  (6, 0, 7, 5, 4, 0, 2, 5, r**5*s - r**4 + r**2 - 1),
  (6, 3, 1, 0, 5, 3, 4, 0, r**5 - r**4 + r**3 - s),
  (7, 1, 2, 0, 5, 1, 3, 0, r**5 - r**3 + r - s),
  (7, 4, 1, 0, 6, 4, 5, 0, r**6 - r**5 + r**4 - s)],
 (0, 1, 1, 1, 1, 1): [(1, 0, 3, 0, 0, 1, 2, 0, r**2*s - r*s + s - 1),
  (1, 0, 4, 1, 0, 2, 3, 1, r**3*s - r**2*s + r*s - 1),
  (1, 0, 5, 2, 0, 3, 4, 2, r**4*s - r**3*s + r**2*s - 1),
  (1, 0, 6, 3, 0, 4, 5, 3, r**5*s - r**4*s + r**3*s - 1),
  (1, 0, 7, 4, 0, 5, 6, 4, r**6*s - r**5*s + r**4*s - 1),
  (2, 0, 6, 0, 0, 2, 4, 0, r**4*s - r**2*s + s - 1),
  (2, 0, 7, 1, 0, 3, 5, 1, r**5*s - r**3*s + r*s - 1),
  (2, 1, 3, 0, 1, 1, 2, 0, r**2*s - r*s - r + s),
  (3, 1, 6, 0, 1, 2, 4, 0, r**4*s - r**2*s - r + s),
  (3, 2, 3, 0, 2, 1, 2, 0, r**2*s - r**2 - r*s + s),
  (4, 2, 6, 0, 2, 2, 4, 0, r**4*s - r**2*s - r**2 + s),
  (4, 3, 3, 0, 3, 1, 2, 0, r**3 - r**2*s + r*s - s),
  (5, 3, 6, 0, 3, 2, 4, 0, r**4*s - r**3 - r**2*s + s),
  (5, 4, 3, 0, 4, 1, 2, 0, r**4 - r**2*s + r*s - s),
  (6, 4, 6, 0, 4, 2, 4, 0, r**4*s - r**4 - r**2*s + s),
  (6, 5, 3, 0, 5, 1, 2, 0, r**5 - r**2*s + r*s - s),
  (7, 5, 6, 0, 5, 2, 4, 0, r**5 - r**4*s + r**2*s - s),
  (7, 6, 3, 0, 6, 1, 2, 0, r**6 - r**2*s + r*s - s)],
 (0, 1, 1, 1, 0, 1): [],
 (0, 0, 1, 0, 0, 1): [],
 (0, 0, 1, 1, 1, 1): [],
 (0, 0, 1, 1, 0, 1): [],
 (1, 1, 1, 1, 1, 1): [],
 (1, 1, 1, 1, 0, 1): [],
 (1, 0, 1, 1, 0, 1): []}